# Practice notebook: check your setup

*Reproducible Workflows with Open Water Data: NASA, NOAA, USGS River Data Access* (CUAHSI)

This short notebook checks that your environment works and gives you a few quick exercises with the tools the course uses: `pandas`, `matplotlib`, `geopandas` and `dataretrieval`. It takes about 10 minutes. Nothing here needs an account or a key.

**Before you start:** select the **Python (open-water-data)** kernel (top right in VS Code). Run the cells in order with Shift+Enter: later cells reuse variables from earlier ones.

**How the exercises work.** Each "Your turn" cell is a starter with `...` blanks and `# TODO:` hints. It stops with an error until you fill the blanks in, which is expected. A collapsed "Show the example answer" follows each one, so try it yourself before you open it.

## 1. Check your environment

This cell prints which Python you're running and tries to import each package the course uses. You want to see the environment folder end in `open-water-data` and a version number next to every package. A line starting with `MISSING` means that package isn't installed in this environment: check that you selected the right kernel and that the environment was built from the course `environment.yml`.

In [1]:
import sys, importlib

print("Python:", sys.version.split()[0])
print("Environment:", sys.prefix)
print()

for name in ["pandas", "numpy", "matplotlib", "geopandas", "requests",
             "dataretrieval", "earthaccess", "xarray", "pynhd"]:
    try:
        module = importlib.import_module(name)
        print(f"ok       {name:15s} {getattr(module, '__version__', '(no version)')}")
    except ImportError:
        print(f"MISSING  {name}")

Python: 3.14.10
Environment: /opt/homebrew/Caskroom/miniforge/base/envs/open-water-data

ok       pandas          3.1.0
ok       numpy           2.5.4
ok       matplotlib      3.11.2
ok       geopandas       1.2.0
ok       requests        2.34.2
ok       dataretrieval   1.4.0
ok       earthaccess     0.19.0
ok       xarray          2026.9.0
ok       pynhd           0.20.0


## 2. Your turn: find the peak with `pandas`

River data usually arrive as a table of dates and values. The cell below builds a small practice table of **made-up** daily discharge values, in cubic feet per second (cfs), for a small imaginary river during a rain event. These are not real USGS data.

In [2]:
import pandas as pd

flow = pd.DataFrame({
    "date": pd.date_range("2025-04-01", periods=10, freq="D"),
    "discharge_cfs": [1200, 1350, 1800, 2600, 4100, 5200, 4700, 3300, 2400, 1900],
})
flow

,date,discharge_cfs
0,2025-04-01,1200
1,2025-04-02,1350
2,2025-04-03,1800
3,2025-04-04,2600
4,2025-04-05,4100
5,2025-04-06,5200
6,2025-04-07,4700
7,2025-04-08,3300
8,2025-04-09,2400
9,2025-04-10,1900


**Task:** find the day with the highest discharge, and print the date and the value.

**What you should see:** a peak of 5200 cfs on 2025-04-06.

In [3]:
# This cell raises an error until the ... is filled in.
peak_idx = ...    # TODO: the row label of the largest discharge (hint: a pandas method named after "index of the maximum")

if peak_idx is ...:
    raise NotImplementedError("Fill in the blank above, then run this cell again.")

peak_row = flow.loc[peak_idx]
print("Peak:", peak_row["discharge_cfs"], "cfs on", peak_row["date"].date())

NotImplementedError: Fill in the blank above, then run this cell again.

<details>
<summary><b>Show the example answer</b></summary>

```python
peak_idx = flow["discharge_cfs"].idxmax()

peak_row = flow.loc[peak_idx]
print("Peak:", peak_row["discharge_cfs"], "cfs on", peak_row["date"].date())
```

</details>

## 3. Your turn: convert units and plot with `matplotlib`

Federal data mix units, so converting is a common first step. One cubic foot is 0.0283168 cubic meters, so a value in cfs times 0.0283168 gives cubic meters per second (m³/s).

**Task:** add a column `discharge_cms` with the flow in m³/s, then plot it against the date, and label the y-axis with the correct units.

**What you should see:** a line with markers that rises to a peak on 6 April (about 147 m³/s) and falls again.

In [ ]:
# This cell raises an error until every ... is filled in.
import matplotlib.pyplot as plt

flow["discharge_cms"] = ...   # TODO: the cfs column times 0.0283168
y_label = ...                 # TODO: a label with the units, e.g. "Discharge (...)"

if flow["discharge_cms"].iloc[0] is ... or y_label is ...:
    raise NotImplementedError("Fill in the blanks above, then run this cell again.")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(flow["date"], flow["discharge_cms"], marker="o")
ax.set_ylabel(y_label)
ax.set_title("Practice river (made-up data)")
fig.autofmt_xdate()
fig.tight_layout()

<details>
<summary><b>Show the example answer</b></summary>

```python
import matplotlib.pyplot as plt

flow["discharge_cms"] = flow["discharge_cfs"] * 0.0283168
y_label = "Discharge (m³/s)"

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(flow["date"], flow["discharge_cms"], marker="o")
ax.set_ylabel(y_label)
ax.set_title("Practice river (made-up data)")
fig.autofmt_xdate()
fig.tight_layout()
```

</details>

## 4. Your turn: reproject a point with `geopandas`

Maps and distances need a coordinate reference system (CRS). Data usually come in longitude and latitude (EPSG:4326), and projected systems use meters. This exercise also confirms that your mapping libraries can find their coordinate database, which is the most common setup problem.

The cell below makes a one-point table for the Ohio River at Louisville, Kentucky, in longitude and latitude.

In [4]:
import geopandas as gpd
from shapely.geometry import Point

gage = gpd.GeoDataFrame(
    {"name": ["Ohio River at Louisville, KY"]},
    geometry=[Point(-85.7991, 38.2803)],   # (longitude, latitude)
    crs="EPSG:4326",
)
print(gage.crs)
gage

EPSG:4326


,name,geometry
0,"Ohio River at Louisville, KY",POINT (-85.7991 38.2803)


**Task:** reproject the point to EPSG:5070 (a projected system for the contiguous US that uses meters), and print its new x and y coordinates.

**What you should see:** the CRS prints as EPSG:5070, and x and y are large numbers in meters (about 882,000 and 1,742,000), not degrees.

In [ ]:
# This cell raises an error until the ... is filled in.
target_crs = ...    # TODO: the code for the projected system named in the task, as a string like "EPSG:1234"

if target_crs is ...:
    raise NotImplementedError("Fill in the blank above, then run this cell again.")

gage_projected = gage.to_crs(target_crs)
point = gage_projected.geometry.iloc[0]
print(gage_projected.crs)
print(f"x = {point.x:,.0f} m, y = {point.y:,.0f} m")

<details>
<summary><b>Show the example answer</b></summary>

```python
target_crs = "EPSG:5070"

gage_projected = gage.to_crs(target_crs)
point = gage_projected.geometry.iloc[0]
print(gage_projected.crs)
print(f"x = {point.x:,.0f} m, y = {point.y:,.0f} m")
```

</details>

## 5. Your turn: download real data with `dataretrieval`

This is the one cell that needs an internet connection. It asks USGS for five days of daily mean discharge at the Ohio River at Louisville, which covers the April 2025 flood. You don't need an account or a key for one small request like this.

Every `waterdata` function returns a pair: a table, and metadata about the request. The USGS codes you need: `00060` is discharge, and `00003` is the daily mean.

**Task:** fill in the two codes, then run the cell.

**What you should see:** 5 rows, one per day from 7 to 11 April 2025, with the highest daily mean on 9 April at about 712,000 ft³/s. Values after the crest are provisional and can be revised, so your numbers may differ slightly. If you get an error about the connection or `429 Too Many Requests`, check your internet connection or wait a few minutes and try again.

In [ ]:
# This cell raises an error until every ... is filled in.
from dataretrieval import waterdata

site = "USGS-03294500"      # Ohio River at Louisville, KY
param = ...                 # TODO: the parameter code for discharge
stat = ...                  # TODO: the statistic code for the daily mean

if param is ... or stat is ...:
    raise NotImplementedError("Fill in the blanks above, then run this cell again.")

daily, md_info = waterdata.get_daily(
    monitoring_location_id=site,
    parameter_code=param,
    statistic_id=stat,
    time="2025-04-07/2025-04-11",
)
daily = daily.sort_values("time")
print(len(daily), "rows")
daily[["time", "value", "unit_of_measure", "approval_status"]]

<details>
<summary><b>Show the example answer</b></summary>

```python
from dataretrieval import waterdata

site = "USGS-03294500"      # Ohio River at Louisville, KY
param = "00060"             # discharge
stat = "00003"              # daily mean

daily, md_info = waterdata.get_daily(
    monitoring_location_id=site,
    parameter_code=param,
    statistic_id=stat,
    time="2025-04-07/2025-04-11",
)
daily = daily.sort_values("time")
print(len(daily), "rows")
daily[["time", "value", "unit_of_measure", "approval_status"]]
```

</details>

## You're set up

If the environment check showed no `MISSING` lines, the reprojection worked, and the USGS download returned five rows, your setup is ready for the course notebooks. If something failed, the troubleshooting section of the setup guide covers the common problems.

Next, open the first lesson notebook, select the same **Python (open-water-data)** kernel, and run its PROJ cell first.